<a href="https://colab.research.google.com/github/karolinakuligowska/Projektowanie_systemow_informatycznych/blob/main/Spinaker_Chatbots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rule-based FAQ chatbot
(works without API keys)

In [1]:
faq = {
    "refund": "Refunds are processed within 7 business days.",
    "delivery": "Standard delivery takes 2-4 business days.",
    "invoice": "Invoices are sent by email after payment confirmation.",
    "premium": "Premium customers receive priority support within 2 hours.",
    "cancel": "Orders can be cancelled within 24 hours of purchase."
}



In [2]:
question = "I am a premium customer. How fast will support answer?"

In [3]:
answer_found = False

for keyword, answer in faq.items():
    if keyword in question.lower():
        print(answer)
        answer_found = True

if not answer_found:
    print("I do not have enough information. Please contact customer support.")

Premium customers receive priority support within 2 hours.


In [5]:
# Test different questions - some will match, some won't
test_questions = [
    "How long does delivery take?",        # matches
    "Can I get a refund?",                  # matches
    "When will my package arrive?"        # no match — 'package' not in FAQ keys!
]

for question in test_questions:
    matched = False
    for key, answer in faq.items():
        if key in question.lower():
            print(f"{question}")
            print(f"{answer}\n")
            matched = True
            break
    if not matched:
        print(f"{question}")
        print("No answer found - keyword matching failed!\n")


How long does delivery take?
Standard delivery takes 2-4 business days.

Can I get a refund?
Refunds are processed within 7 business days.

When will my package arrive?
No answer found - keyword matching failed!



In [6]:
# This is exactly why we need embeddings-based search!

# Better chatbot with embeddings

In [7]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [14]:
knowledge_base = [
    "Refunds are processed within 7 business days.",
    "Standard delivery takes 2-4 business days.",
    "Invoices are sent by email after payment confirmation.",
    "Premium customers receive priority support within 2 hours.",
    "Orders can be cancelled within 24 hours of purchase."
]

model = SentenceTransformer("all-MiniLM-L6-v2")

kb_embeddings = model.encode(knowledge_base)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
customer_question = "My package arrived broken. What should I do?"

question_embedding = model.encode([customer_question])

scores = cosine_similarity(question_embedding, kb_embeddings)[0]


In [15]:
print(f"Customer question: {customer_question}\n")
print("Similarity to each knowledge base entry:\n")

for i, (entry, score) in enumerate(zip(knowledge_base, scores)):
    bar = "█" * int(score * 20)
    print(f"  [{i+1}] {score:.2f}  {bar}")
    print(f"       {entry}")

best_match_index = scores.argmax()
print(f"\nBest match [{best_match_index+1}]: {knowledge_base[best_match_index]}")
print(f" Similarity score: {round(scores[best_match_index], 2)}")


Customer question: My package arrived broken. What should I do?

Similarity to each knowledge base entry:

  [1] 0.14  ██
       Refunds are processed within 7 business days.
  [2] 0.27  █████
       Standard delivery takes 2-4 business days.
  [3] 0.19  ███
       Invoices are sent by email after payment confirmation.
  [4] 0.15  ██
       Premium customers receive priority support within 2 hours.
  [5] 0.39  ███████
       Orders can be cancelled within 24 hours of purchase.

Best match [5]: Orders can be cancelled within 24 hours of purchase.
 Similarity score: 0.38999998569488525


In [11]:
# The model found the right answer using meaning - not keyword matching!

# Another example

In [16]:
def business_chatbot(question):
    question = question.lower()

    if "damaged" in question or "broken" in question:
        return "Please report the damaged product using the complaint form."
    elif "refund" in question:
        return "Refunds are processed within 7 business days."
    elif "premium" in question:
        return "Premium customers receive priority support within 2 hours."
    elif "invoice" in question:
        return "Invoices are sent by email after payment confirmation."
    else:
        return "I do not have enough information to answer this question."

questions = [
    "My product arrived broken. What should I do?",
    "When will I receive my refund?",
    "I am a premium customer. How fast is support?",
    "Where can I download my invoice?",
    "When will my package arrive?"   # intentional no-match — triggers fallback
]



In [17]:
print("Keyword-based chatbot vs. embedding search\n")

for q in questions:
    keyword_answer = business_chatbot(q)
    q_emb = model.encode([q])
    s = cosine_similarity(q_emb, kb_embeddings)[0]
    embedding_answer = knowledge_base[s.argmax()]
    match_score = round(s.max(), 2)

    print(f" {q}")
    print(f"  Keyword bot : {keyword_answer}")
    print(f"  Embedding   : {embedding_answer}  (score: {match_score})")
    print()


Keyword-based chatbot vs. embedding search

 My product arrived broken. What should I do?
  Keyword bot : Please report the damaged product using the complaint form.
  Embedding   : Premium customers receive priority support within 2 hours.  (score: 0.20000000298023224)

 When will I receive my refund?
  Keyword bot : Refunds are processed within 7 business days.
  Embedding   : Refunds are processed within 7 business days.  (score: 0.7099999785423279)

 I am a premium customer. How fast is support?
  Keyword bot : Premium customers receive priority support within 2 hours.
  Embedding   : Premium customers receive priority support within 2 hours.  (score: 0.7699999809265137)

 Where can I download my invoice?
  Keyword bot : Invoices are sent by email after payment confirmation.
  Embedding   : Invoices are sent by email after payment confirmation.  (score: 0.5799999833106995)

 When will my package arrive?
  Keyword bot : I do not have enough information to answer this question.
  Emb

# Discussion:
# Which approach handled better the question: 'When will my package arrive?'

(the keyword bot returns a fallback, but the embedding model finds 'delivery')

## TASK: Improving a Business Chatbot

Work in small groups.

Imagine you are building a chatbot for an online store.

### Step 1

Add 3 new questions and answers to the knowledge base.


### Step 2

Create 5 customer questions that use different wording than the knowledge base.



### Step 3

Test both approaches:

- Keyword-based chatbot
- Embedding-based chatbot

### Discussion

1. Which questions were handled better by the embedding-based chatbot?
2. Which questions were difficult for both approaches?
3. What are the main advantages of keyword matching?
4. What are the main advantages of embeddings?
5. Which approach would you choose for a real customer-service chatbot and why?